# Pack 03 — DFT, FFT, Resolution and Windowing

**Module 05-dsp · Etherstack interview prep · C++ exercises in `code/`, plots and numerical demos here.**

| | |
|---|---|
| Time | ~2 h (45 min reading/demos, 45 min pen-and-paper incl. the 8-point FFT by hand, 30 min C++) |
| You need | Python 3 + NumPy + Matplotlib; CMake + C++17 |
| Signals | synthetic only |

**Etherstack relevance.** FFTs show up everywhere in a radio stack: spectrum/channel scanning, coarse CFO estimation, FFT-based (fast) convolution and channelisers, and OFDM modems (Pack 05). Interviewers love "derive the FFT butterfly" and "why does a window reduce leakage?".

## 1. Concrete first: the DFT is a bank of correlators with rotating phasors

Take N = 16 samples of a cosine that completes **3 cycles** in the frame: x[n] = cos(2π·3·n/16). The DFT bin k *multiplies x[n] by a phasor that makes k turns over the frame and adds up the results*:

$$X[k] = \sum_{n=0}^{N-1} x[n]\, e^{-j2\pi kn/N}.$$

Run the cell: for each k it draws the running sum of the products as a path in the complex plane ("wrapping" picture). When the phasor rotates **with** the signal (k = 3, and k = 13 = −3 mod 16), all products point the same way and the path marches off to |X| = N/2 = 8. For any other k (k = 5 here) the products point in all directions and cancel back to 0.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

N = 16
n = np.arange(N)
x = np.cos(2 * np.pi * 3 * n / N)
fig, axs = plt.subplots(1, 3, figsize=(12, 4))
for ax, k in zip(axs, (3, 5, 13)):
    p = x * np.exp(-2j * np.pi * k * n / N)
    cs = np.cumsum(p)
    ax.plot(np.r_[0, cs.real], np.r_[0, cs.imag], "-o")
    ax.set_title(f"k={k}:  X[k] = {cs[-1]:.2f}")
    ax.set_aspect("equal"); ax.grid(True)
plt.tight_layout(); plt.show()
print(np.round(np.fft.fft(x), 3))

### 1.1 Derivation: why it works (orthogonality)

Correlate with a phasor at bin $m$: put $x[n] = e^{j2\pi m n/N}$ into the definition,

$$X[k] = \sum_{n=0}^{N-1} e^{j2\pi (m-k) n/N}.$$

* If $k = m$: every term is 1, so $X[k] = N$.
* Otherwise it is a geometric series with ratio $r = e^{j2\pi(m-k)/N} \ne 1$ and $r^N = 1$:
$$\sum_{n=0}^{N-1} r^n = \frac{1-r^N}{1-r} = 0.$$

So the $N$ phasors are **orthogonal**, a signal built from them is recovered exactly by correlating, and the inverse is the same sum with the conjugate phasor and a factor $1/N$:

$$x[n] = \frac{1}{N}\sum_{k=0}^{N-1} X[k]\, e^{+j2\pi kn/N}.$$

A cosine is $\tfrac12 e^{j\theta}+\tfrac12 e^{-j\theta}$, so it gives two bins, each of size $N/2$: at $k=m$ and at $N-m$ ("negative frequency").

### 1.2 What the bin index means

Bin $k$ ↔ frequency $k f_s / N$ for $0 \le k < N/2$; bins $N/2 \dots N-1$ are negative frequencies $(k-N) f_s/N$.
**Frequency resolution (bin spacing)** $\Delta f = f_s/N = 1/T_{\text{frame}}$: it depends *only on how long you observed*, not on how many zeros you append.

*Example:* $f_s$ = 48 kHz, $N$ = 1024 → 46.875 Hz per bin. A 1 kHz tone falls at bin 21.33, **between** bins → leakage (§3).

## 2. The FFT: how to compute the DFT in $O(N\log N)$

### 2.1 Cost of the naive DFT
$N$ outputs × $N$ complex multiply-adds = $N^2$. For $N=4096$ that is 16.8 M complex multiplies per frame.

### 2.2 Radix-2 decimation-in-time: derive the butterfly
Let $W_N = e^{-j2\pi/N}$ so $X[k] = \sum_n x[n]W_N^{kn}$. Split the sum into even $n=2r$ and odd $n=2r+1$:

$$X[k] = \sum_{r=0}^{N/2-1} x[2r]\,W_N^{2kr} + W_N^{k}\sum_{r=0}^{N/2-1} x[2r+1]\,W_N^{2kr}.$$

Since $W_N^{2} = e^{-j2\pi/(N/2)} = W_{N/2}$ the two sums are DFTs of length $N/2$:

$$X[k] = E[k] + W_N^{k}\,O[k],\qquad k = 0,\dots,N-1,$$

where $E, O$ are periodic with period $N/2$. So for $k < N/2$ we also know $X[k+N/2]$ for free, using $W_N^{k+N/2} = W_N^k\cdot e^{-j\pi} = -W_N^k$:

$$\boxed{X[k] = E[k] + W_N^k O[k],\qquad X[k+\tfrac N2] = E[k] - W_N^k O[k]}\qquad 0\le k<N/2.$$

That is the **butterfly**: one complex multiply ($W_N^kO[k]$), one add, one subtract produce two outputs. Recurse until length 1 (a length-1 DFT is the sample itself). There are $\log_2 N$ stages, each with $N/2$ butterflies:

$$\text{complex multiplies} = \frac N2 \log_2 N.$$

For $N=4096$: 24 576 vs 16 777 216 — **683× fewer**.

### 2.3 Bit reversal
Splitting even/odd repeatedly sorts the inputs by the bit-reversed index. For $N=8$: input order 0,4,2,6,1,5,3,7. In-place iterative FFTs first permute the data this way and then run stages of size 2, 4, 8, …

In [ ]:
def dft(x):
    N = len(x)
    n = np.arange(N)
    return np.exp(-2j * np.pi * np.outer(n, n) / N) @ x

def fft_rec(x, counter):
    N = len(x)
    if N == 1:
        return x.copy()
    E = fft_rec(x[0::2], counter)
    O = fft_rec(x[1::2], counter)
    W = np.exp(-2j * np.pi * np.arange(N // 2) / N)
    counter[0] += N // 2
    return np.concatenate([E + W * O, E - W * O])

rng = np.random.default_rng(0)
for N in (8, 64, 1024):
    x = rng.standard_normal(N) + 1j * rng.standard_normal(N)
    c = [0]
    X = fft_rec(x, c)
    print(f"N={N:5d}  matches np.fft: {np.allclose(X, np.fft.fft(x))}   DFT mults N^2={N*N:8d}   FFT mults={c[0]:6d}   (N/2)log2N={N//2*int(np.log2(N))}")

### 2.4 The 8-point FFT by hand (worked example you must be able to reproduce)

Input $x = [1,2,3,4,0,0,0,0]$, $W_8^k = e^{-j\pi k/4}$: $W^0=1$, $W^1 = 0.7071-0.7071j$, $W^2=-j$, $W^3 = -0.7071-0.7071j$.

1. **Split.** Even samples $[x_0,x_2,x_4,x_6] = [1,3,0,0]$; odd samples $[x_1,x_3,x_5,x_7] = [2,4,0,0]$.
2. **4-point DFTs** ($W_4 = -j$): $E = [4,\ 1-3j,\ -2,\ 1+3j]$; $O = [6,\ 2-4j,\ -2,\ 2+4j]$.
3. **Combine** with $X[k] = E[k]+W_8^kO[k]$, $X[k+4] = E[k]-W_8^kO[k]$:

| k | $W_8^k O[k]$ | $X[k]$ | $X[k+4]$ |
|---|---|---|---|
| 0 | 6 | 10 | −2 |
| 1 | (0.7071−0.7071j)(2−4j) = −1.4142 − 4.2426j | −0.4142 − 7.2426j | 2.4142 + 1.2426j |
| 2 | (−j)(−2) = 2j | −2 + 2j | −2 − 2j |
| 3 | (−0.7071−0.7071j)(2+4j) = 1.4142 − 4.2426j | 2.4142 − 1.2426j | −0.4142 + 7.2426j |

Checks: (a) $X[0] = \sum x = 10$ ✓. (b) Real input → $X[8-k] = X[k]^*$ ✓ (e.g. $X[5]=X[3]^*$). (c) Parseval: $\sum|X|^2 = 8\sum|x|^2$: left = 240, right = 8·30 ✓.

In [ ]:
x = np.array([1, 2, 3, 4, 0, 0, 0, 0], dtype=complex)
E = np.fft.fft(x[0::2]); O = np.fft.fft(x[1::2])
W = np.exp(-2j * np.pi * np.arange(4) / 8)
print("E =", np.round(E, 4)); print("O =", np.round(O, 4)); print("W*O =", np.round(W * O, 4))
X = np.concatenate([E + W * O, E - W * O])
print("X =", np.round(X, 4))
print("matches np.fft.fft:", np.allclose(X, np.fft.fft(x)))
print("Parseval:", np.sum(np.abs(X) ** 2), "==", 8 * np.sum(np.abs(x) ** 2))

## 3. Leakage and windowing

### 3.1 Concrete first: bin-centred vs off-bin tone
The DFT assumes the frame **repeats forever**. If the tone completes a whole number of cycles per frame the repeated frame is a seamless sinusoid → single bin. If not (bin 10.5) the frame boundary is a discontinuity → energy leaks into all bins.

In [ ]:
N = 128
n = np.arange(N)
def spec_db(x, w=None):
    w = np.ones(len(x)) if w is None else w
    X = np.abs(np.fft.fft(x * w))
    return 20 * np.log10(X / X.max() + 1e-300)

hann = 0.5 - 0.5 * np.cos(2 * np.pi * n / N)           # periodic Hann (for spectral analysis)
tone_on = np.exp(2j * np.pi * 10.0 * n / N)
tone_off = np.exp(2j * np.pi * 10.5 * n / N)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(spec_db(tone_on), label="bin 10.0, rectangular")
ax.plot(spec_db(tone_off), label="bin 10.5, rectangular")
ax.plot(spec_db(tone_off, hann), label="bin 10.5, Hann")
ax.set_ylim(-140, 5); ax.set_xlabel("bin"); ax.set_ylabel("dB rel. peak"); ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()

### 3.2 Derivation: the rectangular window has a Dirichlet-kernel spectrum

Observing $N$ samples is multiplying an infinite tone by a rectangular window. The window's spectrum is the geometric sum

$$W_{\text{rect}}(\omega)=\sum_{n=0}^{N-1}e^{-j\omega n}=\frac{1-e^{-j\omega N}}{1-e^{-j\omega}}
= e^{-j\omega (N-1)/2}\,\frac{\sin(N\omega/2)}{\sin(\omega/2)}.$$

At a distance $\delta$ bins from the tone ($\omega = 2\pi\delta/N$): $|W| \approx N\,\left|\frac{\sin \pi\delta}{\pi\delta}\right|$. Consequences:

* **Main lobe** half-width = 1 bin (first null at $\delta=\pm1$).
* **First sidelobe** at $\delta\approx 1.43$: $\frac{1}{\pi\cdot 1.43}\approx 0.22$ → **−13 dB**; sidelobes then fall only as $1/\delta$ (**−6 dB/octave**).
* Tone **half-way between bins** ($\delta=0.5$): amplitude $\frac{\sin(\pi/2)}{\pi/2} = 0.637$ → **−3.92 dB scalloping loss**.

### 3.3 Hann window: trade main-lobe width for sidelobes

$w[n] = \tfrac12 - \tfrac12\cos(2\pi n/N) = \tfrac12 - \tfrac14 e^{j2\pi n/N} - \tfrac14 e^{-j2\pi n/N}$.
Its spectrum is three shifted Dirichlet kernels with weights $\tfrac12,-\tfrac14,-\tfrac14$ (centres $0,\pm1$ bin). Adjacent-bin contributions *cancel the sidelobes*:

$$|W_{\text{Hann}}(\delta)| \propto \left|\frac{\sin \pi\delta}{\pi\delta\,(1-\delta^2)}\right|.$$

* Main lobe doubles to ±2 bins.
* First sidelobe −31 dB, then falling **18 dB/octave** (the denominator gains a $\delta^2$).
* Scalloping at $\delta=0.5$: $\frac{0.637}{1-0.25}=0.849$ → **−1.42 dB**.
* **Equivalent noise bandwidth** (ENBW) = $N\sum w^2/(\sum w)^2$: rectangular 1.0 bin, Hann $\frac{0.375}{0.25}=1.5$ bins. Noise measured per bin rises by 1.76 dB with Hann.

**Rule:** the window is a trade-off between *frequency selectivity near the tone* (rectangular wins) and *dynamic range far from the tone* (Hann/Blackman-Harris win). Pick Hann for general spectrum views; rectangular only when the signal is exactly periodic in the frame (e.g. OFDM — that is why OFDM needs the cyclic prefix, not a window).

In [ ]:
for name, w in (("rectangular", np.ones(N)), ("Hann", hann)):
    enbw = N * np.sum(w ** 2) / np.sum(w) ** 2
    Xon = np.abs(np.fft.fft(np.exp(2j * np.pi * 10.0 * n / N) * w)).max()
    Xoff = np.abs(np.fft.fft(np.exp(2j * np.pi * 10.5 * n / N) * w)).max()
    far = np.abs(np.fft.fft(np.exp(2j * np.pi * 10.5 * n / N) * w))[70]
    print(f"{name:12s} ENBW={enbw:.3f} bins  scalloping={20*np.log10(Xoff/Xon):6.2f} dB  leakage@bin70={20*np.log10(far/Xoff):7.1f} dB")

### 3.4 Resolution vs zero-padding (a classic trap)

Two tones 0.8 bins apart cannot be separated by a 64-point frame **no matter how many zeros you append**; zero-padding only *interpolates* the same smooth spectrum (it evaluates the DTFT at more points). Only a **longer observation** ($N\uparrow$) narrows the main lobe.

In [ ]:
def two_tone(N, pad):
    n = np.arange(N)
    x = np.exp(2j * np.pi * (10.0 / 64) * n) + np.exp(2j * np.pi * (10.8 / 64) * n)
    w = np.hanning(N)
    X = np.abs(np.fft.fftshift(np.fft.fft(x * w, pad)))
    f = np.fft.fftshift(np.fft.fftfreq(pad))
    return f, X / X.max()

fig, ax = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)
for a, (N_, pad, title) in zip(ax, ((64, 64, "N=64, no pad"), (64, 1024, "N=64, zero-padded to 1024"), (512, 512, "N=512 (8x longer)"))):
    f, X = two_tone(N_, pad)
    a.plot(f, X); a.set_xlim(0.1, 0.2); a.set_title(title); a.grid(True)
plt.tight_layout(); plt.show()

### 3.5 Peak interpolation (fractional-bin estimate)

Near a window's main lobe the log-magnitude is roughly parabolic. Fit a parabola through the peak bin $\beta$ and its neighbours $\alpha$ (left) and $\gamma$ (right) (all in log-magnitude):

$$\delta = \frac12\,\frac{\alpha - \gamma}{\alpha - 2\beta + \gamma},\qquad \hat k = k_{\text{peak}} + \delta.$$

For a Hann window and a tone at 10.3 bins: magnitudes at bins 9, 10, 11 are 0.287, 0.943, 0.721 (from the Hann formula above) → $\delta = 0.316$ → estimate **10.316** (error 0.016 bins). This is how you get sub-bin CFO estimates from an FFT (Etherstack: coarse frequency acquisition).

In [ ]:
def parabolic(Xmag):
    k = int(np.argmax(Xmag)); N = len(Xmag)
    a, b, c = (np.log(Xmag[(k + d) % N] + 1e-300) for d in (-1, 0, 1))
    return (k + 0.5 * (a - c) / (a - 2 * b + c)) % N

for true in (10.0, 10.3, 10.5, 77.7):
    x = np.exp(2j * np.pi * true * n / N) * hann
    print(f"true {true:6.2f}  estimate {parabolic(np.abs(np.fft.fft(x))):8.3f}")

## 4. Misconceptions

| Wrong | Right |
|---|---|
| "Zero-padding increases resolution" | It interpolates; resolution = $f_s/N$ of *real* data |
| "FFT is an approximation of the DFT" | Exactly the same numbers (up to rounding), just faster |
| "A window removes leakage" | It *reshapes* it: lower far sidelobes at the cost of a wider main lobe |
| "Hann window = loses energy so scale doesn't matter" | Coherent gain 0.5 and ENBW 1.5 must be corrected for amplitude / noise-power readings |
| "Bin k at fs·k/N always" | Only for $k<N/2$; upper half are negative frequencies |
| "FFT size must be a power of two" | Radix-2 requires it; mixed-radix/Bluestein don't |
| "The DFT needs the signal to be periodic" | It *assumes* periodic extension — non-periodic ⇒ leakage |

## 5. Etherstack / radio context

* **Channel scan / spectrum view.** A wideband I/Q capture is FFT'd with Hann + averaging to find occupied channels. Resolution must be finer than the 12.5 kHz (or 6.25 kHz) channel raster; $N \ge f_s/\Delta f$.
* **Coarse CFO acquisition.** FFT the signal^M (M-th power of an M-PSK signal) or a preamble tone; peak + parabolic interpolation gives CFO to a fraction of a bin.
* **Fast convolution.** Overlap-save with FFT size $N$ and filter length $L$ costs $O(\log N)$ per sample vs $O(L)$ directly (Pack 04 for FIR, Pack 05 for OFDM).
* **OFDM.** Transmit IFFT, receive FFT; subcarriers are orthogonal because bins are orthogonal (§1.1).

## 6. Exercises

### Part A — pen and paper
1. Compute the 4-point DFT of $[1,0,-1,0]$ using $X[k]=\sum x[n](-j)^{kn}$. Interpret.
2. $f_s$ = 48 kHz. (a) Bin spacing for $N$=1024? (b) Which bin does a 1 kHz tone fall nearest to, and what is the scalloping loss with a rectangular window? (c) To resolve two tones 100 Hz apart with a Hann window (main-lobe full width 4 bins; need ≥ ~2 bins separation) what is the minimum $N$ (power of two)?
3. Show $W_N^{k+N/2} = -W_N^k$ and $W_N^{2} = W_{N/2}$.
4. Count real multiplications for a naive $N=1024$ complex DFT vs radix-2 FFT (one complex multiply = 4 real). 
5. Do the 8-point DFT of $x=[1,1,1,1,0,0,0,0]$ **using the butterfly method** and check Parseval.
6. A real tone sits half-way between bins. Compute scalloping loss for rectangular and Hann windows from the formulae in §3.
7. Write the bit-reversed order for $N=16$.
8. You see a tone at bin 200 in a 256-point FFT of real data sampled at 25.6 kHz. What frequency does bin 200 represent? What is the real frequency (assume the true signal is below fs/2)?

### Part B — C++ (`code/`)
| # | Function | Concept |
|---|---|---|
| B1 | `dft` | definition as correlation, $O(N^2)$ |
| B2 | `bit_reverse` | permutation for in-place FFT |
| B3 | `fft_radix2`, `ifft_radix2` | butterfly |
| B4 | `hann_periodic` | window for spectral analysis |
| B5 | `peak_bin_estimate` | log-parabolic interpolation |

`cmake -S code -B build && cmake --build build && ctest --test-dir build --output-on-failure` (stubs fail; `-DUSE_SOLUTIONS=ON` for reference).

### Part C — interview questions
1. Derive the radix-2 butterfly. How many multiplications for $N=2^m$?
2. Why does a window reduce spectral leakage and what do you pay?
3. Does zero-padding improve frequency resolution? Explain.
4. How would you detect a 12.5 kHz-spaced channel occupancy using an FFT on a 2.4 MS/s capture?
5. What is the FFT's output for a real input and how can you exploit that?

## 7. VERIFY list
* The numbers in §2.4 and §3 (sidelobe −13 dB / −31 dB, scalloping −3.92 / −1.42 dB, ENBW 1.0/1.5) are derived above or are textbook values; **the plotting cells were not executed by the author** — run them and confirm the printed values.
* "Hann far sidelobes fall 18 dB/octave" is for the *periodic* Hann evaluated exactly as in the cell above.
* No standard-specific numbers (symbol rates, deviations) are used in this pack.